In [303]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import RandomizedSearchCV


from sklearn.model_selection import KFold
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

import warnings

In [274]:
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format) 
pd.set_option('display.max_columns', None)              
pd.set_option('display.width', None) 

In [275]:
# Load the dataset
df = pd.read_csv('train_cleaned.csv')
df_test = pd.read_csv('test_cleaned.csv')

In [276]:
df

,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size,IsSuperbowl,IsLaborDay,Ischristmas,IsThanksgiving,year,month,day
0,1,1,2010-02-05,"24,924.50",0,42.31,2.57,0.00,0.00,0.00,0.00,0.00,211.10,8.11,A,151315,0,0,0,0,2010,2,5
1,1,1,2010-02-12,"46,039.49",1,38.51,2.55,0.00,0.00,0.00,0.00,0.00,211.24,8.11,A,151315,1,0,0,0,2010,2,12
2,1,1,2010-02-19,"41,595.55",0,39.93,2.51,0.00,0.00,0.00,0.00,0.00,211.29,8.11,A,151315,0,0,0,0,2010,2,19
3,1,1,2010-02-26,"19,403.54",0,46.63,2.56,0.00,0.00,0.00,0.00,0.00,211.32,8.11,A,151315,0,0,0,0,2010,2,26
4,1,1,2010-03-05,"21,827.90",0,46.50,2.62,0.00,0.00,0.00,0.00,0.00,211.35,8.11,A,151315,0,0,0,0,2010,3,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
421565,45,98,2012-09-28,508.37,0,64.88,4.00,"4,556.61",20.64,1.50,"1,601.01","3,288.25",192.01,8.68,B,118221,0,0,0,0,2012,9,28
421566,45,98,2012-10-05,628.10,0,64.89,3.98,"5,046.74",0.00,18.82,"2,253.43","2,340.01",192.17,8.67,B,118221,0,0,0,0,2012,10,5
421567,45,98,2012-10-12,"1,061.02",0,54.47,4.00,"1,956.28",0.00,7.89,599.32,"3,990.54",192.33,8.67,B,118221,0,0,0,0,2012,10,12
421568,45,98,2012-10-19,760.01,0,56.47,3.97,"2,004.02",0.00,3.18,437.73,"1,537.49",192.33,8.67,B,118221,0,0,0,0,2012,10,19


In [277]:
df.Type = df.Type.map({'A': 1, 'B': 2, 'C': 3})

In [278]:
df

,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size,IsSuperbowl,IsLaborDay,Ischristmas,IsThanksgiving,year,month,day
0,1,1,2010-02-05,"24,924.50",0,42.31,2.57,0.00,0.00,0.00,0.00,0.00,211.10,8.11,1,151315,0,0,0,0,2010,2,5
1,1,1,2010-02-12,"46,039.49",1,38.51,2.55,0.00,0.00,0.00,0.00,0.00,211.24,8.11,1,151315,1,0,0,0,2010,2,12
2,1,1,2010-02-19,"41,595.55",0,39.93,2.51,0.00,0.00,0.00,0.00,0.00,211.29,8.11,1,151315,0,0,0,0,2010,2,19
3,1,1,2010-02-26,"19,403.54",0,46.63,2.56,0.00,0.00,0.00,0.00,0.00,211.32,8.11,1,151315,0,0,0,0,2010,2,26
4,1,1,2010-03-05,"21,827.90",0,46.50,2.62,0.00,0.00,0.00,0.00,0.00,211.35,8.11,1,151315,0,0,0,0,2010,3,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
421565,45,98,2012-09-28,508.37,0,64.88,4.00,"4,556.61",20.64,1.50,"1,601.01","3,288.25",192.01,8.68,2,118221,0,0,0,0,2012,9,28
421566,45,98,2012-10-05,628.10,0,64.89,3.98,"5,046.74",0.00,18.82,"2,253.43","2,340.01",192.17,8.67,2,118221,0,0,0,0,2012,10,5
421567,45,98,2012-10-12,"1,061.02",0,54.47,4.00,"1,956.28",0.00,7.89,599.32,"3,990.54",192.33,8.67,2,118221,0,0,0,0,2012,10,12
421568,45,98,2012-10-19,760.01,0,56.47,3.97,"2,004.02",0.00,3.18,437.73,"1,537.49",192.33,8.67,2,118221,0,0,0,0,2012,10,19


In [279]:

df['MarkDown1'] = df['MarkDown1'].apply(lambda x: 0 if x < 0 else x)
df['MarkDown2'] = df['MarkDown2'].apply(lambda x: 0 if x < 0 else x)
df['MarkDown3'] = df['MarkDown3'].apply(lambda x: 0 if x < 0 else x)
df['MarkDown4'] = df['MarkDown4'].apply(lambda x: 0 if x < 0 else x)
df['MarkDown5'] = df['MarkDown5'].apply(lambda x: 0 if x < 0 else x)

In [280]:
df = df.drop(['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'Unemployment', 'CPI', ], axis=1)

In [281]:
df = df.drop(['IsSuperbowl', 'IsLaborDay', 'IsThanksgiving', 'Ischristmas'], axis=1)

In [282]:
train_data = df[:int(0.7*(len(df)))]
test_data = df[int(0.7*(len(df))):]

target = "Weekly_Sales"
used_cols = [c for c in df.columns.to_list() if c not in [target]] 

X_train = train_data[used_cols]
X_test = test_data[used_cols]
y_train = train_data[target]
y_test = test_data[target]

In [283]:
X_train = X_train.drop(['Date'], axis=1)
X_test = X_test.drop(['Date'], axis=1) 



In [284]:
def wmae_test(test, pred): # WMAE para test set
    weights = X_test['IsHoliday'].apply(lambda is_holiday:4 if is_holiday else 1)
    error = np.sum(weights * np.abs(test - pred), axis=0) / np.sum(weights)
    return error

In [285]:
df

,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,Type,Size,year,month,day
0,1,1,2010-02-05,"24,924.50",0,42.31,2.57,1,151315,2010,2,5
1,1,1,2010-02-12,"46,039.49",1,38.51,2.55,1,151315,2010,2,12
2,1,1,2010-02-19,"41,595.55",0,39.93,2.51,1,151315,2010,2,19
3,1,1,2010-02-26,"19,403.54",0,46.63,2.56,1,151315,2010,2,26
4,1,1,2010-03-05,"21,827.90",0,46.50,2.62,1,151315,2010,3,5
...,...,...,...,...,...,...,...,...,...,...,...,...
421565,45,98,2012-09-28,508.37,0,64.88,4.00,2,118221,2012,9,28
421566,45,98,2012-10-05,628.10,0,64.89,3.98,2,118221,2012,10,5
421567,45,98,2012-10-12,"1,061.02",0,54.47,4.00,2,118221,2012,10,12
421568,45,98,2012-10-19,760.01,0,56.47,3.97,2,118221,2012,10,19


In [286]:
xgb_ = XGBRegressor(n_estimators=100, random_state=42, max_depth=8)

scaler=RobustScaler()

pipe = make_pipeline(scaler,xgb_)

pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_train)

y_pred_test = pipe.predict(X_test)


In [287]:
print("WMAE on test set:", wmae_test(y_test, y_pred_test))

WMAE on test set: 5576.339722550455


In [288]:
params = {
    'n_estimators' : [100,200,300,400], 
    'max_depth' : [7,9],
    'min_child_weight': [7,9], 
    'gamma': [0.1, 0.3]
}

modelo = XGBRegressor(objective ='reg:squarederror', n_jobs=4)

# Definindo k
kfold = KFold(3, shuffle=True, random_state = 42)

# Testando a combinação de parâmetros
grid = RandomizedSearchCV(modelo, params, n_iter=30, cv=kfold, scoring='neg_mean_absolute_percentage_error', n_jobs=-1)
grid_result = grid.fit(X_train, y_train)

# Print do resultado
print("Grid scores on development set:")
means = grid.cv_results_['mean_test_score'].round(5)
stds = grid.cv_results_['std_test_score'].round(5)

for mean, std, params in zip((means), stds, grid.cv_results_['params']):
    print(f'mean:{mean},std:{std},params:{params}')
print()
print(f'Melhor parâmetro:{grid.best_params_}, Score:{grid.best_score_}')

KeyboardInterrupt: 

In [301]:
xgb_ = XGBRegressor(n_estimators= 100, max_depth= 7, gamma= 0.5, random_state=42, eval_metric='mape')

scaler=RobustScaler()

pipe = make_pipeline(scaler,xgb_)

pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_train)

y_pred_test = pipe.predict(X_test)


In [302]:
print("WMAE on test set:", wmae_test(y_test, y_pred_test))

WMAE on test set: 5826.034717827894


In [304]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=8)
scaler_rf = RobustScaler()  

pipe_rf = make_pipeline(scaler_rf, rf)
pipe_rf.fit(X_train, y_train)
y_pred_rf = pipe_rf.predict(X_train)
y_pred_test_rf = pipe_rf.predict(X_test)

In [305]:
wmae_rf = wmae_test(y_test, y_pred_test_rf)
print("WMAE on test set (Random Forest):", wmae_rf)

WMAE on test set (Random Forest): 6413.197143958037
